# Experiment 2.15: Toy tanh gradient-transport probe under orthogonality constraints

This notebook is the analysis counterpart to `run_experiment.py` in the same directory.
It intentionally treats the pair as a **toy, single-seed, single-batch mechanistic probe** rather than
as a definitive demonstration of severe exponential vanishing.

## Pair-level goal

We compare whether different orthogonality-handling strategies change the observed post-training diagnostics in a small tanh network:

- `SGD`
- `Muon`
- `SGD+OrthoPen(0.003)`
- `SGD+OrthoPen(1.0)`
- `SGD+HardOrtho`

The counterpart script is the canonical implementation. This notebook imports and uses the script's returned results instead of re-implementing the experiment core.

## What this notebook actually measures

After **200 fixed training steps** on one random regression batch, the script reports:

- per-layer gradient Frobenius norms
- per-layer `sigma_max(W)`
- per-layer mean `|tanh'(z)|`
- fitted `alpha` from `log(grad_norm)` versus distance from the output layer
- the heuristic proxy `sigma_max(W) * mean|tanh'(z)|`

## What this notebook does **not** establish by itself

- verified convergence
- multiseed uncertainty or robustness
- held-out performance
- exact end-to-end Jacobian transport
- proof that gradients vanish as a specific exponential rate such as `~(0.65)^L`

The proxy is useful for comparing regimes, but it is **not** a full Jacobian or exact product-of-gradients measurement.

In [ ]:
from pathlib import Path
import importlib.util
import platform
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 160)

In [ ]:
def find_pair_dir(start=None):
    start = Path.cwd() if start is None else Path(start)
    relative = Path('experiments/Tier_1_Core_Mechanism_Tests/TANH_VANISHING_ortho_penalty')
    for base in [start, *start.parents]:
        candidate = base / relative
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(
        'Could not locate experiments/Tier_1_Core_Mechanism_Tests/TANH_VANISHING_ortho_penalty '
        f'from working directory {start}'
    )

pair_dir = find_pair_dir()
repo_root = pair_dir.parents[2]
script_path = pair_dir / 'run_experiment.py'
notebook_path = pair_dir / 'run_experiment.ipynb'

spec = importlib.util.spec_from_file_location('tanh_ortho_penalty_experiment', script_path)
if spec is None or spec.loader is None:
    raise ImportError(f'Could not load module spec from {script_path}')
experiment_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(experiment_module)

print('Repository root :', repo_root)
print('Pair directory  :', pair_dir)
print('Script path     :', script_path)
print('Notebook path   :', notebook_path)

## Reproducibility, entrypoint, and runtime

This cell runs the canonical script logic with `verbose=False`, captures the returned structured results,
and logs the main reproducibility information needed to re-run the same toy study.

In [ ]:
notebook_start = time.perf_counter()
experiment = experiment_module.run_experiment(verbose=False)
notebook_runtime = time.perf_counter() - notebook_start
config = experiment['config']

print('Counterpart script        :', experiment['identity']['script_path'])
print('Recommended CLI entrypoint:', experiment['identity']['recommended_run_command'])
print('Current working directory :', Path.cwd())
print('Python                    :', sys.version.split()[0])
print('Platform                  :', platform.platform())
print('Notebook-observed runtime :', f'{notebook_runtime:.2f}s')
print('Script-reported runtime   :', f"{experiment['runtime_seconds']:.2f}s")
print('Methods                   :', ', '.join(experiment['methods']))
print('Depths                    :', config['depths'])
print('Seed policy               :', experiment['seed_policy'])
print('Dataset reuse             :', experiment['dataset']['reuse_policy'])
print('Loss definition           :', config['loss_definition'])

config

### Reproducibility note

- The dataset is one fixed random batch reused for every depth and method.
- At a fixed depth, all methods start from the **same** initialization seed.
- The notebook and script are now aligned because the notebook consumes the script's returned data rather than duplicating the experiment core.

## Summary table by depth and method

The table below is the main compact summary of what the current implementation computes.
It keeps the proxy explicitly labeled as a proxy, and it includes the fitted `alpha` value without overstating what that fit proves.

In [ ]:
summary_df = pd.DataFrame(experiment['summary_rows']).sort_values(['depth', 'method']).reset_index(drop=True)
summary_view = summary_df[[
    'depth', 'method', 'alpha', 'loss', 'mean_sigma_max', 'mean_tanh_deriv',
    'mean_proxy_multiplier', 'ratio', 'proxy_all_below_1', 'proxy_all_above_1'
]].copy()
summary_view.round(4)

### Table interpretation

A serious reading of the table is:

- **Hard orthogonality** and the **strong penalty** mainly show up as `sigma_max` control.
- **Weak penalty** stays close to SGD in this toy run.
- **Muon** reaches lower loss here while allowing larger singular values.
- The fitted `alpha` values should be read as a compact attenuation diagnostic, not as proof of a universal exponential law.

In [ ]:
method_order = experiment['methods']
palette = {
    'SGD': '#1f77b4',
    'Muon': '#ff7f0e',
    'SGD+OrthoPen(0.003)': '#2ca02c',
    'SGD+OrthoPen(1.0)': '#d62728',
    'SGD+HardOrtho': '#9467bd',
}

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8), sharex=True)

for method in method_order:
    subset = summary_df[summary_df['method'] == method].sort_values('depth')
    color = palette[method]
    axes[0].plot(subset['depth'], subset['alpha'], marker='o', linewidth=2, color=color, label=method)
    axes[1].plot(subset['depth'], subset['mean_sigma_max'], marker='o', linewidth=2, color=color, label=method)
    axes[2].plot(subset['depth'], subset['mean_proxy_multiplier'], marker='o', linewidth=2, color=color, label=method)

axes[0].axhline(1.0, color='black', linestyle='--', linewidth=1)
axes[1].axhline(1.0, color='black', linestyle='--', linewidth=1)
axes[2].axhline(1.0, color='black', linestyle='--', linewidth=1)

axes[0].set_title('Alpha vs depth')
axes[0].set_xlabel('Depth')
axes[0].set_ylabel('Alpha')

axes[1].set_title('Mean sigma_max vs depth')
axes[1].set_xlabel('Depth')
axes[1].set_ylabel('Mean sigma_max')

axes[2].set_title('Mean proxy vs depth')
axes[2].set_xlabel('Depth')
axes[2].set_ylabel("Mean sigma_max * mean|tanh'|")

for ax in axes:
    ax.set_xticks(sorted(summary_df['depth'].unique()))

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.08))
fig.tight_layout()
plt.show()

### Aggregate plot interpretation

These plots support a narrower claim than the original over-strong narrative:

- The orthogonality-constrained regimes keep `sigma_max` close to `1`, especially `SGD+HardOrtho`.
- In this run, those same regimes also push the **proxy** below `1`.
- However, the fitted `alpha` values remain much closer to `1` than an aggressive "severe vanishing" story would predict.
- Therefore the current pair supports a **proxy-level compensation story** more clearly than it supports a strong claim about dramatic exponential decay.

In [ ]:
selected_depth = max(config['depths'])
layer_rows = []
for method in method_order:
    row = experiment['results_by_depth_method'][selected_depth][method]
    for layer_idx, (grad_norm, sigma_max, tanh_deriv, proxy) in enumerate(
        zip(row['grad_norms'], row['sigma_maxes'], row['mean_tanh_derivs'], row['proxy_multipliers'])
    ):
        layer_rows.append({
            'depth': selected_depth,
            'method': method,
            'layer': layer_idx,
            'grad_norm': grad_norm,
            'sigma_max': sigma_max,
            'mean_tanh_deriv': tanh_deriv,
            'proxy_multiplier': proxy,
        })

layer_df = pd.DataFrame(layer_rows)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), sharex=True)
for method in method_order:
    subset = layer_df[layer_df['method'] == method].sort_values('layer')
    color = palette[method]
    axes[0].plot(subset['layer'], subset['grad_norm'], marker='o', linewidth=2, color=color, label=method)
    axes[1].plot(subset['layer'], subset['proxy_multiplier'], marker='o', linewidth=2, color=color, label=method)

axes[0].set_title(f'Layerwise gradient norms at depth={selected_depth}')
axes[0].set_xlabel('Layer index (0 = deepest)')
axes[0].set_ylabel('Gradient Frobenius norm')

axes[1].axhline(1.0, color='black', linestyle='--', linewidth=1)
axes[1].set_title(f'Layerwise proxy values at depth={selected_depth}')
axes[1].set_xlabel('Layer index (0 = deepest)')
axes[1].set_ylabel("sigma_max * mean|tanh'|")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.08))
fig.tight_layout()
plt.show()

### Deep-case layerwise interpretation

The depth-8 view is useful because it shows the mechanism at the layer level rather than only through depth-averaged summaries.
In the current implementation, the strongest visual separation is usually in the **proxy values** and in the constraint-induced control of `sigma_max`, not in a dramatic collapse of fitted `alpha`.

## Script diagnostics table

The script also returns a small set of **toy-scope checks**. These checks are intentionally narrow:
for example, they ask whether the hard-orthogonal regime keeps mean `sigma_max` near `1` and whether its mean proxy is below `1`.
They are not presented as proof of a full gradient-transport theory.

In [ ]:
diagnostics_df = pd.DataFrame(experiment['diagnostics']['per_depth'])
diagnostics_df.round(4)

## Calibrated conclusion

For this first implementation pass, the notebook and script now tell a consistent and narrower story:

1. In this toy setup, **hard orthogonality** and the **strong orthogonality penalty** keep `sigma_max` near `1`.
2. Those same regimes also tend to place the heuristic proxy `sigma_max(W) * mean|tanh'(z)|` below `1`.
3. **SGD** and especially **Muon** allow larger singular values, and their mean proxy values remain above `1` in this run.
4. The observed `alpha` values do **not** justify presenting the pair as a definitive demonstration of severe exponential vanishing such as `alpha < 0.7`.
5. The honest scientific status of the pair is therefore: **a deterministic toy mechanistic probe whose current evidence is strongest at the proxy-diagnostic level**.

### Recommended next verification if stronger claims are needed

- multiseed runs with uncertainty bars
- trajectory logging over training, not only post-training snapshots
- a more direct end-to-end gradient transport or Jacobian-based metric